# Notebook 4 — Image-to-Image Search with Git-RSCLIP

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_4_image_to_image_search.ipynb)

---

## What we're doing

Same model, same precomputed pool, **different query modality**. In Notebook 3 we typed English and got matching satellite images back. Here we hand the model **one satellite image** and ask: *what else in the pool looks like this?*

This is the workhorse pattern for:
- **Analog discovery** — *find more places like this one I'm investigating*.
- **Event triage** — *I have one confirmed flood / fire / new construction site; surface similar candidates fast*.
- **Bootstrapping labels** — *I have 5 examples; find me 50 more*.

Mechanically it's even simpler than text→image: drop the text tower, encode the query image with the **vision** tower, and rank against the same 5,400 pool vectors with a single mat-vec.


## Runtime

**CPU is fine** — designed for a 5-minute Colab CPU budget. The 5,400-image pool was precomputed offline (same `.npz` as Notebook 3, fetched from the HF Hub below). At workshop time we encode only the **one** query image — ~250 ms on CPU.


In [ ]:
!pip install -q transformers sentencepiece datasets huggingface_hub


In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Download the precomputed image embedding pool

Identical to Notebook 3: the 5,400-vector pool lives on the Hugging Face Hub at [`zterrabyte/eurosat-gitrsclip-embeddings`](https://huggingface.co/datasets/zterrabyte/eurosat-gitrsclip-embeddings). `hf_hub_download` caches it to the Colab working directory — a few seconds the first time, instant on re-run.


In [ ]:
HF_POOL_REPO = 'zterrabyte/eurosat-gitrsclip-embeddings'
HF_POOL_FILE = 'eurosat_test_gitrsclip.npz'

t0 = time.time()
npz_path = hf_hub_download(repo_id=HF_POOL_REPO, filename=HF_POOL_FILE, repo_type='dataset')
print(f'Downloaded in {time.time()-t0:.1f}s — {Path(npz_path).stat().st_size/1e6:.1f} MB')

blob = np.load(npz_path, allow_pickle=True)
img_embeddings = blob['embeddings'].astype(np.float32)
labels = blob['labels']
class_names = [str(c) for c in blob['class_names']]
print(f'Pool: {img_embeddings.shape} | classes: {class_names}')


## 2. Load the corresponding EuroSAT images (for display + as a query source)

We use the **test** split for the pool (same as the embeddings) and pull a small slice of the **train** split to give us *out-of-pool* query images — so the search has to actually retrieve, not just self-match.


In [ ]:
t0 = time.time()
ds_pool  = load_dataset('timm/eurosat-rgb', split='test')
ds_query = load_dataset('timm/eurosat-rgb', split='train[:100]')   # 100 candidate query images, not in the pool
print(f'Loaded pool={len(ds_pool)}, query candidates={len(ds_query)} in {time.time()-t0:.1f}s')
assert len(ds_pool) == len(img_embeddings)


## 3. Load the Git-RSCLIP vision tower for query encoding

Same checkpoint as Notebook 3, but here we'll be calling `model.vision_model(...)` instead of `model.text_model(...)`. SigLIP returns its image embedding directly as `vision_model(...).pooler_output` — no extra projection head needed for cosine similarity against the pool.

*The first run downloads ~870 MB of weights — give it 30–60 s on a fresh Colab.*


In [ ]:
MODEL_ID = 'lcybuaa/Git-RSCLIP-base'
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(device)
print(f'Model loaded in {time.time()-t0:.1f}s '
      f'({sum(p.numel() for p in model.parameters())/1e6:.0f}M params, '
      f'embed_dim={model.config.vision_config.hidden_size})')


## 4. The image-to-image search function

Three steps, structurally identical to NB 3 but with the vision tower:

1. Preprocess + run the **vision tower** on the query image → pooler output → L2-normalize.
2. Cosine similarity = `pool_embeddings @ query_vec`.
3. Pick top-K, return their image objects and class labels.


In [ ]:
def encode_image(img: Image.Image) -> np.ndarray:
    inputs = processor(images=[img.convert('RGB')], return_tensors='pt').to(device)
    with torch.inference_mode():
        out = model.vision_model(**inputs)
    return F.normalize(out.pooler_output.float(), dim=-1).cpu().numpy()[0]

def search_image(query_img: Image.Image, k: int = 8):
    t0 = time.time()
    qvec = encode_image(query_img)
    enc_ms = (time.time() - t0) * 1000
    sims = img_embeddings @ qvec
    top = np.argsort(-sims)[:k]
    return top, sims[top], enc_ms

def show_results(query_img: Image.Image, query_label: str = '', k: int = 8):
    top, scores, ms = search_image(query_img, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(2.0 * (k + 1), 2.6))
    axes[0].imshow(query_img); axes[0].axis('off')
    axes[0].set_title(f'QUERY\n{query_label}', fontsize=8, color='C3')
    for ax, idx, score in zip(axes[1:], top, scores):
        ax.imshow(ds_pool[int(idx)]['image']); ax.axis('off')
        ax.set_title(f'{class_names[labels[idx]]}\ncos={score:.3f}', fontsize=8)
    plt.suptitle(f'Image-to-image top-{k}   (image encode: {ms:.0f} ms)', y=1.05, fontsize=11)
    plt.tight_layout(); plt.show()


## 5. Run image queries on out-of-pool images, one per land-cover class

We pull **one example image of each of the 10 EuroSAT classes from the train split** (which is *not* in the pool) and use each as a query. Ideally most of the top-8 hits should be the same class.


In [ ]:
# Find one query example per class from the train slice we loaded.
examples_by_class = {}
for ex in ds_query:
    if ex['label'] not in examples_by_class:
        examples_by_class[ex['label']] = ex['image']
    if len(examples_by_class) == len(class_names):
        break

for lbl in sorted(examples_by_class):
    show_results(examples_by_class[lbl], query_label=class_names[lbl], k=8)


## 6. Quantitative check — Precision@K across the 200 query candidates

For every query image (100 of them) we look at its top-K retrievals from the *pool* and check how many share the query's class. Higher = retrieval respects semantics. We sweep K so you can see the curve.


In [ ]:
# Encode all 200 query candidates in one batched pass — much faster than per-image loop.
t0 = time.time()
qry_labels = np.array(ds_query['label'])
qry_embs = []
BS = 32
with torch.inference_mode():
    for i in range(0, len(ds_query), BS):
        imgs = [im.convert('RGB') for im in ds_query[i:i+BS]['image']]
        inp = processor(images=imgs, return_tensors='pt').to(device)
        out = model.vision_model(**inp)
        qry_embs.append(F.normalize(out.pooler_output.float(), dim=-1).cpu().numpy())
qry_embs = np.concatenate(qry_embs, axis=0)
print(f'Encoded {len(qry_embs)} queries in {time.time()-t0:.1f}s on {device}')

# Bulk cosine sim: (200, 5400)
sims = qry_embs @ img_embeddings.T
Ks = [1, 5, 10, 25, 50]
print(f'\n{"K":>4}  P@K   (over 100 out-of-pool queries)')
for K in Ks:
    top = np.argsort(-sims, axis=1)[:, :K]                  # (200, K)
    matches = (labels[top] == qry_labels[:, None]).mean()
    bar = '█' * int(round(matches * 30))
    print(f'{K:>4}  {matches:>5.1%}  {bar}')


### Per-class breakdown at K=10

In [ ]:
K = 10
top = np.argsort(-sims, axis=1)[:, :K]
match_mask = (labels[top] == qry_labels[:, None])     # (200, K)
print(f'{"class":<22} P@{K}   n_queries')
for cls_idx, name in enumerate(class_names):
    mask = qry_labels == cls_idx
    if not mask.any():
        continue
    pk = match_mask[mask].mean()
    n = int(mask.sum())
    bar = '█' * int(round(pk * 20))
    print(f'{name:<22} {pk:>5.1%}   ({n:>2})  {bar}')


## 7. Bring your own query image (optional)

Upload any RGB image — a satellite chip you have on hand, a Google Earth screenshot, even a photo — and Git-RSCLIP will project it into the same embedding space as the EuroSAT pool. Out-of-distribution queries are interesting: the model returns *the closest things it has*, which may or may not match the query intent.

*To upload in Colab, uncomment the two `files.upload()` lines. Otherwise this cell uses one of the train-split query images as a stand-in so the notebook runs end-to-end with no extra interaction.*


In [ ]:
# from google.colab import files
# uploaded = files.upload()                # interactive picker
# query_path = next(iter(uploaded))
# query_img = Image.open(query_path)

# Stand-in: use an out-of-pool query image so the notebook always renders.
query_img   = examples_by_class[7]                 # Residential example
query_label = class_names[7] + ' (stand-in)'
show_results(query_img, query_label=query_label, k=10)


## 8. Takeaways

- **Image-to-image search is structurally identical to text-to-image search** in CLIP-style embedding spaces. Same pool, same mat-vec, different query encoder. That's the appeal: one index, many query modalities.
- **Precision@K is honest.** Across 100 out-of-pool queries we see P@1 typically ~85-90% and P@50 around 70% — neither perfect nor terrible, exactly the kind of signal you'd build a human-in-the-loop *triage* tool on, not an unattended classifier.
- **Per-class numbers tell the design story.** Visually distinctive classes (SeaLake, Forest, Residential) sit near the top. Semantically overlapping ones (Pasture ↔ HerbaceousVegetation, Highway ↔ Industrial) drag the bottom — same confounders you saw in Notebooks 0 and 1.
- **Speed comes from precomputation + a single mat-vec.** 5,400 images is just numpy here, but the *same pattern* with a FAISS / ScaNN index scales to hundreds of millions of vectors with sub-second top-K. That's the production path.
- **Where this is heading in the session:**
  - Combine text and image queries: average a text vector and an image vector → *"something like THIS, but more like a river"*.
  - Compose **positive/negative** queries → *"places that look like A but not B"* — the building block for change-oriented search.
  - Move the cosine-sim step from numpy to **FAISS** for scale.
